# RF-DETR Faces Local Workflow

This notebook runs the local Mac workflow for the RF-DETR faces test benchmark. The CLI tools in `src/rfdetr_faces/` are the source of truth; this notebook is a guided wrapper for setup checks, extraction, inference, and COCO export.


## Dataset Policy

The cleaned Roboflow dataset from these target videos is a test-only benchmark. Do not use these images for training, fine-tuning, threshold selection, or augmentation experiments. Train on broader/general face datasets, then evaluate against this target-domain test set.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "rfdetr_faces").exists():
            return candidate
    raise RuntimeError("Could not find repo root from notebook location.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

VIDEO_DIR = REPO_ROOT / "face-trimmed-videos"
FRAMES_DIR = REPO_ROOT / "data" / "frames"
PREDICTIONS_PATH = REPO_ROOT / "data" / "predictions" / "predictions.jsonl"
MODEL_PATH = REPO_ROOT / "models" / "checkpoint_best_ema.pth"
PREVIEW_DIR = REPO_ROOT / "runs" / "previews"
COCO_EXPORT_DIR = REPO_ROOT / "data" / "roboflow-export"

print(f"Repo root: {REPO_ROOT}")
print(f"Model exists: {MODEL_PATH.exists()} - {MODEL_PATH}")


In [ ]:
import cv2
import torch
from rfdetr import RFDETRLarge

print(f"OpenCV: {cv2.__version__}")
print(f"Torch: {torch.__version__}")
print(f"MPS available: {torch.backends.mps.is_available()}")
print(f"RFDETRLarge import: {RFDETRLarge}")


In [ ]:
def read_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]


frame_rows = read_jsonl(FRAMES_DIR / "metadata.jsonl")
prediction_rows = read_jsonl(PREDICTIONS_PATH)
annotation_path = COCO_EXPORT_DIR / "train" / "_annotations.coco.json"

print(f"Videos: {len(list(VIDEO_DIR.glob('*.mp4'))) if VIDEO_DIR.exists() else 0}")
print(f"Extracted frame rows: {len(frame_rows)}")
print(f"Prediction rows: {len(prediction_rows)}")
print(f"Prediction boxes: {sum(len(row.get('detections', [])) for row in prediction_rows)}")
print(f"COCO annotations file exists: {annotation_path.exists()}")


## Pipeline Controls

Keep these flags `False` unless you intentionally want to run that step. Inference can take time because it loads RF-DETR and processes all extracted frames.


In [ ]:
RUN_EXTRACTION = False
RUN_INFERENCE = False
RUN_EXPORT = False

EXTRACTION_FPS = 3
PREDICTION_THRESHOLD = 0.25
PREDICTION_BATCH_SIZE = 4
INCLUDE_EMPTY_IN_EXPORT = True


In [ ]:
if RUN_EXTRACTION:
    subprocess.run(
        [
            "uv",
            "run",
            "rfdetr-faces",
            "extract-frames",
            "--fps",
            str(EXTRACTION_FPS),
            "--overwrite",
        ],
        cwd=REPO_ROOT,
        check=True,
    )
else:
    print("Skipping extraction. Set RUN_EXTRACTION = True to run it.")


In [ ]:
if RUN_INFERENCE:
    subprocess.run(
        [
            "uv",
            "run",
            "rfdetr-faces",
            "predict-faces",
            "--weights",
            str(MODEL_PATH),
            "--threshold",
            str(PREDICTION_THRESHOLD),
            "--batch-size",
            str(PREDICTION_BATCH_SIZE),
            "--preview-dir",
            str(PREVIEW_DIR),
        ],
        cwd=REPO_ROOT,
        check=True,
    )
else:
    print("Skipping inference. Set RUN_INFERENCE = True to run it.")


In [ ]:
prediction_rows = read_jsonl(PREDICTIONS_PATH)
empty_images = sum(1 for row in prediction_rows if not row.get("detections"))
box_count = sum(len(row.get('detections', [])) for row in prediction_rows)

print(f"Prediction rows: {len(prediction_rows)}")
print(f"Prediction boxes: {box_count}")
print(f"Images without detections: {empty_images}")

if prediction_rows:
    prediction_rows[0]


In [ ]:
if RUN_EXPORT:
    command = [
        "uv",
        "run",
        "rfdetr-faces",
        "export-coco",
        "--overwrite",
    ]
    if INCLUDE_EMPTY_IN_EXPORT:
        command.append("--include-empty")
    subprocess.run(command, cwd=REPO_ROOT, check=True)
else:
    print("Skipping COCO export. Set RUN_EXPORT = True to run it.")


In [ ]:
if annotation_path.exists():
    coco_data = json.loads(annotation_path.read_text())
    print(f"COCO images: {len(coco_data['images'])}")
    print(f"COCO annotations: {len(coco_data['annotations'])}")
    print(f"COCO categories: {coco_data['categories']}")
else:
    print(f"COCO annotation file not found: {annotation_path}")


## Roboflow Versioning

The Roboflow dataset created from this export should keep all images in the test split. Suggested version note: `target-video-test-3fps-clean`. Use no augmentations for this benchmark version.


## Legacy Colab Workflow

The original Colab notebook trained and evaluated RF-DETR using Google Drive paths and inline install cells. Those cells were removed from this local notebook to avoid mixing training, upload, and benchmark evaluation in one artifact. Keep any future training notebook separate from this test benchmark workflow.
